# Proyecto 3 - RNN Vanilla para autocompletar codigo C

Entrenamiento de un modelo de lenguaje a nivel de **caracter** con una
**RNN Vanilla** () en Keras/TensorFlow. El corpus se
construye muestreando 150 funciones en C del dataset crudo con
.

Arquitectura (literal del ejemplo del profesor, paginas 6124-6143):



Perdida: .
Optimizador: Adam(1e-3).


In [1]:
import os, sys
PROJECT_ROOT = "/home/jojo/develop/academic/IA/3-RNN"
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print("cwd:", os.getcwd())
import json, random
from pathlib import Path
import numpy as np
import tensorflow as tf
import matplotlib
matplotlib.use("Agg")  # headless
import matplotlib.pyplot as plt

tf.keras.utils.set_random_seed(42)
random.seed(42)
np.random.seed(42)

print("TF:", tf.__version__)
print("numpy:", np.__version__)
print("python:", sys.version_info[:3])

cwd: /home/jojo/develop/academic/IA/3-RNN


2026-06-01 15:32:39.319168: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-01 15:32:39.323817: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-01 15:32:39.382012: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


2026-06-01 15:32:40.723191: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


TF: 2.16.1
numpy: 1.26.4
python: (3, 11, 9)


## 1. Muestreo del dataset

El corpus crudo vive en . Aplicamos heuristicas de
estilo (ASCII, sin , longitud razonable) y nos quedamos
con 150 funciones representativas.


In [2]:
!python src/sample_clean.py --input dataset/funciones.c --out-dir dataset/sample --target 150

Reading first 20,000,000 bytes of dataset/funciones.c ...


Loaded 19,996,114 chars.


Found 26,439 candidate signatures.


Scanned, kept 19565 candidates. Rejected: {'has_main': 4939, 'length=2362': 4, 'length=1540': 2, 'length=3750': 1, 'non_ascii': 34, 'length=2648': 1, 'length=13925': 1, 'length=2350': 4, 'length=2836': 1, 'length=2271': 1, 'length=1862': 1, 'length=5670': 1, 'length=6379': 3, 'length=3157': 1, 'length=2764': 2, 'length=2312': 2, 'length=1808': 1, 'length=2366': 1, 'length=1882': 1, 'length=6063': 1, 'length=1511': 3, 'length=1633': 2, 'length=1506': 3, 'length=1874': 2, 'length=1709': 1, 'length=1931': 2, 'length=4974': 1, 'length=2085': 2, 'length=2647': 4, 'length=2579': 1, 'length=263953': 1, 'length=3593': 2, 'length=3216': 1, 'length=8141': 1, 'length=1705': 5, 'length=1707': 2, 'length=1806': 2, 'length=1556': 2, 'length=2399': 1, 'length=4156': 1, 'length=1630': 3, 'length=2810': 1, 'length=1672': 4, 'length=2761': 1, 'length=1600': 3, 'length=3160': 1, 'length=2040': 1, 'length=2429': 1, 'length=1875': 3, 'length=1885': 2, 'length=2136': 1, 'length=3001': 1, 'length=2885': 1, '

Wrote 150 functions to dataset/sample/funciones.c (69,874 bytes)
Reporte en dataset/sample/REPORTE.md


## 2. Preprocesamiento (char-level)

Vocabulario = caracteres unicos en el corpus. Cada muestra es una
ventana de 32 caracteres y el objetivo es el siguiente caracter en
cada posicion (igual que el ejemplo del profesor).

In [3]:
!python src/preprocess.py --input dataset/sample/funciones.c --output-dir dataset/processed --block-size 32

vocab_size=94 | corpus_chars=69,874 | num_windows=69,842
X.shape=(69842, 32) Y.shape=(69842, 32) dtype=int64
Saved to dataset/processed


In [4]:
meta = json.loads(Path("dataset/processed/meta.json").read_text())
X = np.load("dataset/processed/X.npy")
Y = np.load("dataset/processed/Y.npy")
print("vocab_size =", meta["vocab_size"])
print("block_size =", meta["block_size"])
print("corpus_chars =", meta["corpus_chars"])
print("num_windows =", meta["num_windows"])
print("X.shape =", X.shape, " Y.shape =", Y.shape)

vocab_size = 94
block_size = 32
corpus_chars = 69874
num_windows = 69842
X.shape = (69842, 32)  Y.shape = (69842, 32)


## 3. Definicion del modelo

Tres capas, ~17 K parametros. Es la **RNN Vanilla** que pide la
actividad - explicitamente **no** LSTM ni GRU.


In [5]:
from tensorflow.keras import Input, Sequential
from tensorflow.keras.layers import Dense, Embedding, SimpleRNN, TimeDistributed

EMBED_DIM = 48
HIDDEN = 64
VOCAB = meta["vocab_size"]

model = Sequential([
    Input(shape=(meta["block_size"],)),
    Embedding(VOCAB, EMBED_DIM),
    SimpleRNN(HIDDEN, activation="tanh", return_sequences=True),
    TimeDistributed(Dense(VOCAB)),
])
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 32, 48)         │         4,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 32, 64)         │         7,232 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 32, 94)         │         6,110 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,854 (69.74 KB)

 Trainable params: 17,854 (69.74 KB)

 Non-trainable params: 0 (0.00 B)

## 4. Entrenamiento (80 epocas)

Submuestreo a 20 K ventanas para mantener el entrenamiento rapido en
CPU (sin GPU en este equipo). Loss esperada: ~3.0 -> ~1.1.


In [6]:
MAX_WINDOWS = 20_000
if len(X) > MAX_WINDOWS:
    idx = np.random.RandomState(42).permutation(len(X))[:MAX_WINDOWS]
    idx.sort()
    Xs, Ys = X[idx], Y[idx]
else:
    Xs, Ys = X, Y
print("Train set:", Xs.shape, Ys.shape)

history = model.fit(
    Xs, Ys,
    epochs=50,
    batch_size=128,
    verbose=0,
)
losses = [float(x) for x in history.history["loss"]]
print(f"loss: {losses[0]:.4f} -> {losses[-1]:.4f}")

Train set: (20000, 32) (20000, 32)


loss: 3.0657 -> 1.0978


## 5. Curva de perdida

La perdida por caracter baja de ~3 a ~1 en 80 epocas. La entropia
inicial  es el peor caso (prediccion uniforme sobre
todo el vocabulario), asi que 1.0 es una mejora sustancial.

In [7]:
plt.figure(figsize=(8, 4))
plt.plot(losses, label="train loss")
plt.xlabel("epoca")
plt.ylabel("loss")
plt.title("Curva de perdida - SimpleRNN char-level")
plt.grid(True, linestyle=":", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

/tmp/ipykernel_69949/2024941348.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Guardar el modelo para el servidor

Guardamos  +  en . El servidor
stdio (F6) y la API FastAPI (F5) los cargan desde ahi.


In [8]:
Path("models").mkdir(exist_ok=True)
model.save("models/rnn_v1.keras")
deploy_meta = dict(meta)
deploy_meta["model_path"] = str(Path("models/rnn_v1.keras").resolve())
deploy_meta["params"] = {"embed_dim": EMBED_DIM, "hidden": HIDDEN, "epochs": 50, "batch_size": 128, "lr": 1e-3}
Path("models/rnn_v1.meta.json").write_text(json.dumps(deploy_meta, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved models/rnn_v1.keras + rnn_v1.meta.json")

Saved models/rnn_v1.keras + rnn_v1.meta.json


## 7. Generacion autorregresiva (sanity check)

Cargamos el modelo y muestreamos el siguiente caracter 
veces. La temperatura controla el equilibrio determinismo / azar.


In [9]:
from src.predict import RNNModel, complete

rnn = RNNModel.load(Path("models/rnn_v1.keras"))
for prompt in ["int sum", "void print", "for (int", "if (a"]:
    out = complete(rnn, prompt, max_new=50, temperature=0.4, seed=42)
    print(repr(prompt), "->", repr(prompt + out))

'int sum' -> "int sum[mid] < nums[0] == 0)\n  return '!';\n if (strcmp(na"


'void print' -> 'void printf("FAILED: %s\\n"); } else {\n                    re'


'for (int' -> "for (int source <= '9') {\n        return binarySearch(arr,"


'if (a' -> 'if (array[i];\n  }\n    if (array[mid] < nums[0])\n       '


## 8. Conclusion

La red aprende patrones lexicos/sintacticos de C (parentesis,
palabras clave, tipos). **No** produce codigo ejecutable - es un
modelo didactico que demuestra:

1. Como  mapea caracteres a vectores.
2. Como  propaga contexto temporal.
3. Como  aplica la misma capa densa en
   cada paso.

Para pasar de demo a utilidad habria que multiplicar el corpus por
~1000, entrenar >500 epocas o migrar a LSTM/GRU. La actividad
explicita RNN Vanilla, asi que con esto cumplimos.